# GNNExplainer Variance Analysis

This notebook computes explanation variance for GNNExplainer on the test split.
It follows the same graph selection logic as `experiments.ipynb`:
`graph_scope='test_split'`, `first_graph_per_component=True`, `component_key='compound'`.

Outputs are written to `results/variance/`.


In [1]:
import sys
import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import (
    ModelConfig,
    ModelMode,
    ModelReturnType,
    ModelTaskLevel,
)


def _resolve_project_root() -> Path:
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd / "gnn4nmr", cwd.parent / "gnn4nmr"]
    for candidate in candidates:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(
        f"Could not resolve project root from cwd={cwd}. Expected a folder with scripts/ and notebooks/."
    )


PROJECT_ROOT = _resolve_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"

for path in [str(PROJECT_ROOT), str(SCRIPTS_DIR), str(NOTEBOOK_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)


from scripts.explainer.experiments_evaluation import (
    build_default_context,
    select_scope_graph_indices,
    first_graph_indices_per_component,
)
from scripts.explainer.explainer_utils import (
    NodeTypeRegressionWrapper,
    get_device,
    load_config,
    load_stats,
    build_dataset,
    load_trained_model,
    heterodata_to_dicts,
)


try:
    from feature_visualization import get_feature_names as _get_feature_names
except Exception:
    _get_feature_names = None


def set_all_seeds(seed: int) -> None:
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def clone_edge_attr_dict(edge_attr_dict):
    if edge_attr_dict is None:
        return None
    return {
        edge_type: (attrs.clone() if attrs is not None else None)
        for edge_type, attrs in edge_attr_dict.items()
    }


def tensor_to_numpy(value) -> np.ndarray:
    if value is None:
        return np.asarray([], dtype=float)
    if isinstance(value, torch.Tensor):
        arr = value.detach().cpu().numpy().astype(float)
    else:
        arr = np.asarray(value, dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    return arr


def edge_type_sort_key(edge_type):
    src, rel, dst = edge_type
    return (str(src), str(rel), str(dst))


def extract_feature_vectors(explanation, feature_node_types=("H", "C")):
    vectors = {}
    node_mask_dict = getattr(explanation, "node_mask_dict", {}) or {}
    for node_type in feature_node_types:
        mask = node_mask_dict.get(node_type)
        arr = tensor_to_numpy(mask)
        if arr.ndim == 0:
            arr = arr.reshape(1)
        if arr.ndim == 1:
            vec = arr.reshape(-1)
        elif arr.ndim >= 2:
            arr2 = arr.reshape(arr.shape[0], -1)
            vec = arr2.mean(axis=0)
        else:
            vec = np.asarray([], dtype=float)
        vectors[node_type] = np.asarray(vec, dtype=float).reshape(-1)
    return vectors


def build_edge_vector(explanation, edge_index_dict):
    edge_mask_dict = getattr(explanation, "edge_mask_dict", {}) or {}
    values = []

    for edge_type in sorted(edge_index_dict.keys(), key=edge_type_sort_key):
        num_edges = int(edge_index_dict[edge_type].size(1))
        mask = edge_mask_dict.get(edge_type)
        mask_arr = tensor_to_numpy(mask).reshape(-1)

        for pos in range(num_edges):
            if pos < int(mask_arr.size):
                value = float(mask_arr[pos])
                values.append(value if np.isfinite(value) else 0.0)
            else:
                values.append(0.0)

    return np.asarray(values, dtype=float)


def pad_and_stack(vectors):
    arrs = [np.asarray(v, dtype=float).reshape(-1) for v in vectors]
    if not arrs:
        return np.zeros((0, 0), dtype=float)
    max_dim = max(int(a.size) for a in arrs)
    if max_dim <= 0:
        return np.zeros((len(arrs), 0), dtype=float)
    mat = np.zeros((len(arrs), max_dim), dtype=float)
    for i, arr in enumerate(arrs):
        if arr.size > 0:
            mat[i, : int(arr.size)] = np.where(np.isfinite(arr), arr, 0.0)
    return mat


def vector_variance(vectors):
    if len(vectors) < 2:
        return np.asarray([], dtype=float)
    mat = pad_and_stack(vectors)
    if mat.shape[1] == 0:
        return np.asarray([], dtype=float)
    return np.var(mat, axis=0, ddof=0)


def mean_dim_variance(vectors):
    dim_var = vector_variance(vectors)
    if dim_var.size == 0:
        return float("nan")
    return float(np.mean(dim_var))


def collect_candidate_nodes(dataset, eval_graph_indices, node_types=("H", "C"), explanation_type="phenomenon"):
    rows = []
    explanation_type = str(explanation_type).strip().lower()

    for graph_idx in sorted(int(i) for i in eval_graph_indices):
        data = dataset[graph_idx]

        for node_type in node_types:
            if node_type not in data.node_types:
                continue

            x = data[node_type].x
            num_nodes = int(x.size(0))
            if num_nodes <= 0:
                continue

            y_tensor = getattr(data[node_type], "y", None)
            y_flat = y_tensor.reshape(-1) if isinstance(y_tensor, torch.Tensor) else None

            if explanation_type == "phenomenon" and y_flat is None:
                continue

            for node_idx in range(num_nodes):
                target_value = float("nan")
                if y_flat is not None and node_idx < int(y_flat.numel()):
                    target_value = float(y_flat[node_idx].item())

                if explanation_type == "phenomenon" and not np.isfinite(target_value):
                    continue

                rows.append(
                    {
                        "graph_idx": int(graph_idx),
                        "target_node_type": str(node_type),
                        "node_idx": int(node_idx),
                        "target_value": float(target_value),
                    }
                )

    return rows


def safe_feature_name(node_type: str, feature_idx: int) -> str:
    if callable(_get_feature_names):
        try:
            names = list(_get_feature_names(node_type))
            if 0 <= int(feature_idx) < len(names):
                return str(names[int(feature_idx)])
        except Exception:
            pass
    return f"feature_{node_type}_{int(feature_idx)}"


print(f"cwd: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Models dir: {MODELS_DIR}")
print(f"Data dir: {DATA_DIR}")


cwd: /home/bautz/gnn4nmr/notebooks
Project root: /home/bautz/gnn4nmr
Notebook dir: /home/bautz/gnn4nmr/notebooks
Models dir: /home/bautz/gnn4nmr/models
Data dir: /home/bautz/gnn4nmr/data


In [2]:
def _pick_default_file(directory: Path, pattern: str, preferred: str):
    preferred_path = directory / preferred
    if preferred_path.exists():
        return preferred
    matches = sorted(p.name for p in directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern!r} found in {directory}")
    return matches[0]


MODEL_FILE = _pick_default_file(MODELS_DIR, "*.pt", "GraphConv_best_model.pt")
DATA_FILE = _pick_default_file(DATA_DIR, "*.pkl", "all_graphs_with_length.pkl")
SPLIT_FILE = "models/graph_split.pkl"

GRAPH_SCOPE = "test_split"
FIRST_GRAPH_PER_COMPONENT = True
COMPONENT_KEY = "compound"

NODE_TYPES = ("H", "C")
REPEATS = 5
SEEDS = [0, 1, 2, 3, 4]
assert len(SEEDS) == REPEATS, "SEEDS and REPEATS mismatch"

# Optional limits (0 means no limit)
MAX_GRAPHS = 3  # set to 3 for smoke test
MAX_NODES_PER_GRAPH = 0

GNN_EPOCHS = 200
GNN_LR = 0.01
GNN_EXPLANATION_TYPE = "phenomenon"

OUTPUT_DIR = PROJECT_ROOT / "results" / "variance"
PLOTS_DIR = OUTPUT_DIR / "plots"

TOP_N_FEATURES_PLOT = 30
FIG_DPI = 180

COLOR_BLUE = "#004e9f"
COLOR_YELLOW = "#fcba00"
COLOR_GREY = "#909085"

print("Variance config loaded")
print(f"MODEL_FILE={MODEL_FILE}")
print(f"DATA_FILE={DATA_FILE}")
print(f"SPLIT_FILE={SPLIT_FILE}")
print(f"GRAPH_SCOPE={GRAPH_SCOPE}")
print(f"FIRST_GRAPH_PER_COMPONENT={FIRST_GRAPH_PER_COMPONENT}, COMPONENT_KEY={COMPONENT_KEY}")
print(f"NODE_TYPES={NODE_TYPES}, REPEATS={REPEATS}, SEEDS={SEEDS}")
print(f"MAX_GRAPHS={MAX_GRAPHS}, MAX_NODES_PER_GRAPH={MAX_NODES_PER_GRAPH}")
print(f"GNN settings: epochs={GNN_EPOCHS}, lr={GNN_LR}, explanation_type={GNN_EXPLANATION_TYPE}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")


Variance config loaded
MODEL_FILE=GraphConv_best_model.pt
DATA_FILE=all_graphs_with_length.pkl
SPLIT_FILE=models/graph_split.pkl
GRAPH_SCOPE=test_split
FIRST_GRAPH_PER_COMPONENT=True, COMPONENT_KEY=compound
NODE_TYPES=('H', 'C'), REPEATS=5, SEEDS=[0, 1, 2, 3, 4]
MAX_GRAPHS=3, MAX_NODES_PER_GRAPH=0
GNN settings: epochs=200, lr=0.01, explanation_type=phenomenon
OUTPUT_DIR=/home/bautz/gnn4nmr/results/variance


In [3]:
context = build_default_context(
    project_root=PROJECT_ROOT,
    model_file=MODEL_FILE,
    data_file=DATA_FILE,
    split_file=SPLIT_FILE,
    output_dir=str(OUTPUT_DIR),
)

device = get_device(None)
config = load_config(str(context.config_path))
norm_stats, edge_stats = load_stats(str(context.norm_stats_path), str(context.edge_stats_path))
dataset = build_dataset(str(context.data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
base_model = load_trained_model(str(context.model_path), config, device)

scope_graph_indices = select_scope_graph_indices(
    dataset=dataset,
    split_path=context.split_path,
    graph_scope=GRAPH_SCOPE,
)

if FIRST_GRAPH_PER_COMPONENT:
    eval_graph_indices = first_graph_indices_per_component(
        dataset=dataset,
        graph_indices=scope_graph_indices,
        component_key=COMPONENT_KEY,
    )
else:
    eval_graph_indices = [int(i) for i in scope_graph_indices]

if int(MAX_GRAPHS) > 0:
    eval_graph_indices = eval_graph_indices[: int(MAX_GRAPHS)]

candidate_nodes = collect_candidate_nodes(
    dataset=dataset,
    eval_graph_indices=eval_graph_indices,
    node_types=NODE_TYPES,
    explanation_type=GNN_EXPLANATION_TYPE,
)

if int(MAX_NODES_PER_GRAPH) > 0:
    grouped = defaultdict(list)
    for row in candidate_nodes:
        grouped[(int(row["graph_idx"]), str(row["target_node_type"]))].append(row)

    reduced = []
    rng = random.Random(0)
    for key in sorted(grouped.keys()):
        rows = grouped[key]
        if len(rows) > int(MAX_NODES_PER_GRAPH):
            pick_idx = sorted(rng.sample(range(len(rows)), int(MAX_NODES_PER_GRAPH)))
            rows = [rows[i] for i in pick_idx]
        reduced.extend(rows)

    candidate_nodes = sorted(
        reduced,
        key=lambda x: (int(x["graph_idx"]), str(x["target_node_type"]), int(x["node_idx"])),
    )


graph_meta_rows = []
for graph_idx in eval_graph_indices:
    data = dataset[int(graph_idx)]
    num_nodes = sum(int(data[nt].x.size(0)) for nt in data.node_types)
    num_edges = sum(int(store.edge_index.size(1)) for store in data.edge_stores)
    compound = dataset.nx_graphs[int(graph_idx)].graph.get(COMPONENT_KEY, "unknown")
    graph_meta_rows.append(
        {
            "graph_idx": int(graph_idx),
            "compound": str(compound),
            "graph_num_nodes": int(num_nodes),
            "graph_num_edges": int(num_edges),
        }
    )

graph_meta_df = pd.DataFrame(graph_meta_rows).sort_values("graph_idx").reset_index(drop=True)
graph_meta_map = {
    int(row["graph_idx"]): row
    for row in graph_meta_df.to_dict(orient="records")
}

graph_cache = {}
for graph_idx in eval_graph_indices:
    data = dataset[int(graph_idx)].to(device)
    x_dict_raw, edge_index_dict, edge_attr_dict_raw, y_dict_raw = heterodata_to_dicts(data)

    x_dict = {nt: feat.clone() for nt, feat in x_dict_raw.items()}
    edge_attr_dict = clone_edge_attr_dict(edge_attr_dict_raw)
    y_dict = {
        nt: (val.clone() if isinstance(val, torch.Tensor) else val)
        for nt, val in y_dict_raw.items()
    }

    graph_cache[int(graph_idx)] = {
        "x_dict": x_dict,
        "edge_index_dict": edge_index_dict,
        "edge_attr_dict": edge_attr_dict,
        "y_dict": y_dict,
    }

print(f"Device: {device}")
print(f"Dataset size: {len(dataset)}")
print(f"Graphs in scope before component filter: {len(scope_graph_indices)}")
print(f"Graphs selected for evaluation: {len(eval_graph_indices)}")
print(f"Candidate nodes selected: {len(candidate_nodes)}")
print("Node counts by target type:")
print(pd.Series([row["target_node_type"] for row in candidate_nodes]).value_counts())

display(graph_meta_df.head(10))


Device: cuda
Dataset size: 940
Graphs in scope before component filter: 150
Graphs selected for evaluation: 3
Candidate nodes selected: 42
Node counts by target type:
H    26
C    16
Name: count, dtype: int64


,graph_idx,compound,graph_num_nodes,graph_num_edges
0,50,6,14,30
1,170,21,20,42
2,270,31,11,22


In [ ]:
model_config = ModelConfig(
    mode=ModelMode.regression,
    task_level=ModelTaskLevel.node,
    return_type=ModelReturnType.raw,
)

explanation_rows = []
failure_rows = []

feature_vectors_by_node = defaultdict(lambda: {nt: [] for nt in NODE_TYPES})
edge_vectors_by_node = defaultdict(list)
seed_runs_by_node = defaultdict(set)
target_value_by_node = {}

for seed in SEEDS:
    set_all_seeds(seed)

    explainers = {}
    for target_node_type in NODE_TYPES:
        wrapped_model = NodeTypeRegressionWrapper(base_model, target_node_type)
        explainers[target_node_type] = Explainer(
            model=wrapped_model,
            algorithm=GNNExplainer(epochs=int(GNN_EPOCHS), lr=float(GNN_LR)),
            explanation_type=str(GNN_EXPLANATION_TYPE),
            model_config=model_config,
            node_mask_type="attributes",
            edge_mask_type="object",
        )

    print(f"Seed {seed}: explaining {len(candidate_nodes)} nodes")

    for row in candidate_nodes:
        graph_idx = int(row["graph_idx"])
        target_node_type = str(row["target_node_type"])
        node_idx = int(row["node_idx"])
        target_value = float(row.get("target_value", float("nan")))

        cache_item = graph_cache.get(graph_idx)
        if cache_item is None:
            failure_rows.append(
                {
                    "seed": int(seed),
                    "graph_idx": int(graph_idx),
                    "target_node_type": target_node_type,
                    "node_idx": int(node_idx),
                    "error": "graph_cache_missing",
                }
            )
            continue

        x_dict = {nt: feat.clone() for nt, feat in cache_item["x_dict"].items()}
        edge_index_dict = cache_item["edge_index_dict"]
        edge_attr_dict = clone_edge_attr_dict(cache_item["edge_attr_dict"])
        y_dict = cache_item["y_dict"]

        gnn_target = None
        if str(GNN_EXPLANATION_TYPE).strip().lower() == "phenomenon":
            y_tensor = y_dict.get(target_node_type)
            if y_tensor is None:
                failure_rows.append(
                    {
                        "seed": int(seed),
                        "graph_idx": int(graph_idx),
                        "target_node_type": target_node_type,
                        "node_idx": int(node_idx),
                        "error": "missing_target_tensor",
                    }
                )
                continue

            y_tensor = y_tensor.reshape(-1)
            if node_idx < 0 or node_idx >= int(y_tensor.size(0)):
                failure_rows.append(
                    {
                        "seed": int(seed),
                        "graph_idx": int(graph_idx),
                        "target_node_type": target_node_type,
                        "node_idx": int(node_idx),
                        "error": "node_idx_out_of_bounds",
                    }
                )
                continue
            if torch.isnan(y_tensor[node_idx]):
                failure_rows.append(
                    {
                        "seed": int(seed),
                        "graph_idx": int(graph_idx),
                        "target_node_type": target_node_type,
                        "node_idx": int(node_idx),
                        "error": "target_is_nan",
                    }
                )
                continue

            gnn_target = y_tensor

        try:
            try:
                explanation = explainers[target_node_type](
                    x_dict,
                    edge_index_dict,
                    edge_attr_dict=edge_attr_dict,
                    target=gnn_target,
                    index=node_idx,
                )
            except Exception as inner_exc:
                if str(GNN_EXPLANATION_TYPE).strip().lower() == "phenomenon" and gnn_target is not None:
                    explanation = explainers[target_node_type](
                        x_dict,
                        edge_index_dict,
                        edge_attr_dict=edge_attr_dict,
                        target=gnn_target[node_idx],
                        index=node_idx,
                    )
                else:
                    raise inner_exc

            feature_vectors = extract_feature_vectors(
                explanation,
                feature_node_types=NODE_TYPES,
            )
            feature_h = np.asarray(feature_vectors.get("H", np.asarray([], dtype=float)), dtype=float).reshape(-1)
            feature_c = np.asarray(feature_vectors.get("C", np.asarray([], dtype=float)), dtype=float).reshape(-1)
            edge_vector = build_edge_vector(explanation, edge_index_dict).reshape(-1)

            meta = graph_meta_map.get(graph_idx, {})
            explanation_rows.append(
                {
                    "seed": int(seed),
                    "graph_idx": int(graph_idx),
                    "compound": str(meta.get("compound", "unknown")),
                    "graph_num_nodes": int(meta.get("graph_num_nodes", 0)),
                    "graph_num_edges": int(meta.get("graph_num_edges", 0)),
                    "target_node_type": target_node_type,
                    "node_idx": int(node_idx),
                    "target_value": float(target_value),
                    "feature_h_dim": int(feature_h.size),
                    "feature_c_dim": int(feature_c.size),
                    "edge_dim": int(edge_vector.size),
                    "feature_h_vector": json.dumps(feature_h.tolist()),
                    "feature_c_vector": json.dumps(feature_c.tolist()),
                    "edge_vector": json.dumps(edge_vector.tolist()),
                }
            )

            key = (int(graph_idx), target_node_type, int(node_idx))
            feature_vectors_by_node[key]["H"].append(feature_h)
            feature_vectors_by_node[key]["C"].append(feature_c)
            edge_vectors_by_node[key].append(edge_vector)
            seed_runs_by_node[key].add(int(seed))
            target_value_by_node[key] = float(target_value)

        except Exception as exc:
            failure_rows.append(
                {
                    "seed": int(seed),
                    "graph_idx": int(graph_idx),
                    "target_node_type": target_node_type,
                    "node_idx": int(node_idx),
                    "error": str(exc),
                }
            )


explanation_runs_df = pd.DataFrame(explanation_rows)
failures_df = pd.DataFrame(failure_rows)

feature_dim_rows = []
node_variance_rows = []

for key in sorted(seed_runs_by_node.keys()):
    graph_idx, target_node_type, node_idx = key
    meta = graph_meta_map.get(graph_idx, {})

    feature_var_mean = {}

    for feature_source_type in NODE_TYPES:
        vectors = feature_vectors_by_node[key].get(feature_source_type, [])
        dim_var = vector_variance(vectors)

        if dim_var.size > 0:
            for feature_idx, var_value in enumerate(dim_var.tolist()):
                feature_dim_rows.append(
                    {
                        "graph_idx": int(graph_idx),
                        "compound": str(meta.get("compound", "unknown")),
                        "target_node_type": str(target_node_type),
                        "node_idx": int(node_idx),
                        "feature_source_type": str(feature_source_type),
                        "feature_idx": int(feature_idx),
                        "feature_variance": float(var_value),
                    }
                )
            feature_var_mean[feature_source_type] = float(np.mean(dim_var))
        else:
            feature_var_mean[feature_source_type] = float("nan")

    edge_var_mean = mean_dim_variance(edge_vectors_by_node.get(key, []))

    finite_vals = [
        v
        for v in [feature_var_mean.get("H"), feature_var_mean.get("C"), edge_var_mean]
        if np.isfinite(v)
    ]
    overall_var_mean = float(np.mean(finite_vals)) if finite_vals else float("nan")

    node_variance_rows.append(
        {
            "graph_idx": int(graph_idx),
            "compound": str(meta.get("compound", "unknown")),
            "graph_num_nodes": int(meta.get("graph_num_nodes", 0)),
            "graph_num_edges": int(meta.get("graph_num_edges", 0)),
            "target_node_type": str(target_node_type),
            "node_idx": int(node_idx),
            "target_value": float(target_value_by_node.get(key, float("nan"))),
            "seed_runs": int(len(seed_runs_by_node.get(key, set()))),
            "feature_var_mean_H": float(feature_var_mean.get("H", float("nan"))),
            "feature_var_mean_C": float(feature_var_mean.get("C", float("nan"))),
            "edge_var_mean": float(edge_var_mean),
            "overall_var_mean": float(overall_var_mean),
        }
    )


feature_variance_dim_df = pd.DataFrame(feature_dim_rows)
node_variance_df = pd.DataFrame(node_variance_rows)

if feature_variance_dim_df.empty:
    feature_variance_by_target_df = pd.DataFrame(
        columns=[
            "target_node_type",
            "feature_source_type",
            "feature_idx",
            "feature_variance_mean",
            "feature_variance_median",
            "feature_variance_std",
            "n_node_entries",
            "feature_name",
            "feature_label",
        ]
    )
else:
    feature_variance_by_target_df = (
        feature_variance_dim_df.groupby(
            ["target_node_type", "feature_source_type", "feature_idx"],
            as_index=False,
        )
        .agg(
            feature_variance_mean=("feature_variance", "mean"),
            feature_variance_median=("feature_variance", "median"),
            feature_variance_std=("feature_variance", lambda s: float(np.std(s.to_numpy(dtype=float), ddof=0))),
            n_node_entries=("feature_variance", "size"),
        )
        .sort_values(["target_node_type", "feature_variance_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )

    feature_variance_by_target_df["feature_name"] = feature_variance_by_target_df.apply(
        lambda row: safe_feature_name(str(row["feature_source_type"]), int(row["feature_idx"])),
        axis=1,
    )
    feature_variance_by_target_df["feature_label"] = feature_variance_by_target_df.apply(
        lambda row: f"{row['feature_source_type']}:{row['feature_name']}",
        axis=1,
    )


if node_variance_df.empty:
    graph_edge_variance_df = pd.DataFrame(
        columns=[
            "graph_idx",
            "compound",
            "graph_num_nodes",
            "graph_num_edges",
            "explained_nodes",
            "graph_edge_var_mean",
            "graph_edge_var_median",
            "graph_edge_var_std",
        ]
    )
else:
    graph_edge_variance_df = (
        node_variance_df.groupby("graph_idx", as_index=False)
        .agg(
            compound=("compound", "first"),
            graph_num_nodes=("graph_num_nodes", "first"),
            graph_num_edges=("graph_num_edges", "first"),
            explained_nodes=("node_idx", "count"),
            graph_edge_var_mean=("edge_var_mean", "mean"),
            graph_edge_var_median=("edge_var_mean", "median"),
            graph_edge_var_std=("edge_var_mean", lambda s: float(np.std(pd.to_numeric(s, errors="coerce").dropna().to_numpy(dtype=float), ddof=0))),
        )
        .sort_values("graph_idx")
        .reset_index(drop=True)
    )

print("Computation finished")
print(f"explanation_runs_df: {explanation_runs_df.shape}")
print(f"node_variance_df: {node_variance_df.shape}")
print(f"feature_variance_dim_df: {feature_variance_dim_df.shape}")
print(f"feature_variance_by_target_df: {feature_variance_by_target_df.shape}")
print(f"graph_edge_variance_df: {graph_edge_variance_df.shape}")
print(f"failures_df: {failures_df.shape}")

display(node_variance_df.head(10))
display(feature_variance_by_target_df.head(10))
display(graph_edge_variance_df.head(10))


Seed 0: explaining 42 nodes


/home/bautz/anaconda3/envs/gnn4nmr-lx/lib/python3.10/site-packages/torch_geometric/nn/dense/linear.py:127: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:217.)
  return F.linear(x, self.weight, self.bias)
/home/bautz/anaconda3/envs/gnn4nmr-lx/lib/python3.10/site-packages/torch/nn/modules/linear.py:125: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(t

Seed 1: explaining 42 nodes


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

explanation_runs_path = OUTPUT_DIR / "explanation_runs_raw.csv"
node_variance_path = OUTPUT_DIR / "node_level_variance.csv"
feature_variance_dim_path = OUTPUT_DIR / "feature_variance_dim.csv"
feature_variance_target_path = OUTPUT_DIR / "feature_variance_by_target.csv"
graph_edge_variance_path = OUTPUT_DIR / "graph_edge_variance.csv"
failures_path = OUTPUT_DIR / "failed_explanations.csv"
run_config_path = OUTPUT_DIR / "run_config.json"

explanation_runs_df.to_csv(explanation_runs_path, index=False)
node_variance_df.to_csv(node_variance_path, index=False)
feature_variance_dim_df.to_csv(feature_variance_dim_path, index=False)
feature_variance_by_target_df.to_csv(feature_variance_target_path, index=False)
graph_edge_variance_df.to_csv(graph_edge_variance_path, index=False)
failures_df.to_csv(failures_path, index=False)

run_config = {
    "model_file": str(MODEL_FILE),
    "data_file": str(DATA_FILE),
    "split_file": str(SPLIT_FILE),
    "graph_scope": str(GRAPH_SCOPE),
    "first_graph_per_component": bool(FIRST_GRAPH_PER_COMPONENT),
    "component_key": str(COMPONENT_KEY),
    "node_types": [str(nt) for nt in NODE_TYPES],
    "repeats": int(REPEATS),
    "seeds": [int(s) for s in SEEDS],
    "max_graphs": int(MAX_GRAPHS),
    "max_nodes_per_graph": int(MAX_NODES_PER_GRAPH),
    "gnn_epochs": int(GNN_EPOCHS),
    "gnn_lr": float(GNN_LR),
    "gnn_explanation_type": str(GNN_EXPLANATION_TYPE),
    "output_dir": str(OUTPUT_DIR),
    "plots_dir": str(PLOTS_DIR),
    "counts": {
        "graphs_selected": int(len(eval_graph_indices)),
        "candidate_nodes": int(len(candidate_nodes)),
        "run_rows": int(len(explanation_runs_df)),
        "node_variance_rows": int(len(node_variance_df)),
        "feature_variance_rows": int(len(feature_variance_by_target_df)),
        "graph_edge_rows": int(len(graph_edge_variance_df)),
        "failed_rows": int(len(failures_df)),
    },
    "artifacts": {
        "explanation_runs_raw": str(explanation_runs_path),
        "node_level_variance": str(node_variance_path),
        "feature_variance_dim": str(feature_variance_dim_path),
        "feature_variance_by_target": str(feature_variance_target_path),
        "graph_edge_variance": str(graph_edge_variance_path),
        "failed_explanations": str(failures_path),
        "run_config": str(run_config_path),
    },
}

with run_config_path.open("w", encoding="utf-8") as handle:
    json.dump(run_config, handle, indent=2)

print("Saved variance artifacts:")
print(f"  {explanation_runs_path}")
print(f"  {node_variance_path}")
print(f"  {feature_variance_dim_path}")
print(f"  {feature_variance_target_path}")
print(f"  {graph_edge_variance_path}")
print(f"  {failures_path}")
print(f"  {run_config_path}")


In [ ]:
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"

# --- Feature variance bar plots, separated by prediction target ---
for target in ["H", "C"]:
    target_df = feature_variance_by_target_df[
        feature_variance_by_target_df["target_node_type"] == target
    ].copy()

    if target_df.empty:
        print(f"Skip feature variance plot for target={target}: no rows")
        continue

    plot_df = target_df.sort_values("feature_variance_mean", ascending=False).head(int(TOP_N_FEATURES_PLOT)).copy()
    plot_df = plot_df.sort_values("feature_variance_mean", ascending=True)

    colors = [
        COLOR_BLUE if str(src) == "H" else COLOR_YELLOW
        for src in plot_df["feature_source_type"].tolist()
    ]

    fig_h = max(4.5, 0.34 * len(plot_df) + 1.5)
    fig, ax = plt.subplots(figsize=(12, fig_h))

    ax.barh(
        plot_df["feature_label"],
        plot_df["feature_variance_mean"],
        color=colors,
        edgecolor="none",
        alpha=0.9,
    )

    ax.set_title(f"Feature variance by prediction target (Target={target})", loc="left")
    ax.set_xlabel("Mean feature variance across repeated explanations")
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.3, color=COLOR_GREY)

    from matplotlib.patches import Patch

    legend_handles = [
        Patch(facecolor=COLOR_BLUE, edgecolor="none", label="H features"),
        Patch(facecolor=COLOR_YELLOW, edgecolor="none", label="C features"),
    ]
    ax.legend(handles=legend_handles, loc="best", frameon=False)

    fig.tight_layout()
    out_path = PLOTS_DIR / f"feature_variance_target_{target}.png"
    fig.savefig(out_path, dpi=int(FIG_DPI), bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()


# --- Edge variance histogram (mean edge variance per graph) ---
edge_vals = pd.to_numeric(graph_edge_variance_df.get("graph_edge_var_mean"), errors="coerce")
edge_vals = edge_vals[np.isfinite(edge_vals)]

if edge_vals.empty:
    print("Skip edge variance histogram: no finite graph_edge_var_mean values")
else:
    bins = min(30, max(5, int(np.sqrt(len(edge_vals)))))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(edge_vals, bins=bins, color=COLOR_GREY, alpha=0.9, edgecolor="white", linewidth=0.8)
    ax.set_title("Histogram of mean edge variance per graph", loc="left")
    ax.set_xlabel("Mean edge variance per graph")
    ax.set_ylabel("Number of graphs")
    ax.grid(axis="y", alpha=0.3, color=COLOR_GREY)

    fig.tight_layout()
    hist_path = PLOTS_DIR / "edge_variance_histogram.png"
    fig.savefig(hist_path, dpi=int(FIG_DPI), bbox_inches="tight")
    print(f"Saved: {hist_path}")
    plt.show()


# --- Edge variance scatter (variance vs graph size = node count) ---
scatter_df = graph_edge_variance_df.copy()
scatter_df["graph_num_nodes"] = pd.to_numeric(scatter_df["graph_num_nodes"], errors="coerce")
scatter_df["graph_edge_var_mean"] = pd.to_numeric(scatter_df["graph_edge_var_mean"], errors="coerce")
scatter_df = scatter_df[
    np.isfinite(scatter_df["graph_num_nodes"]) & np.isfinite(scatter_df["graph_edge_var_mean"])
].copy()

if scatter_df.empty:
    print("Skip edge variance scatter: no finite rows")
else:
    fig, ax = plt.subplots(figsize=(9, 5.5))

    ax.scatter(
        scatter_df["graph_num_nodes"],
        scatter_df["graph_edge_var_mean"],
        color=COLOR_BLUE,
        edgecolor=COLOR_GREY,
        linewidth=0.7,
        alpha=0.85,
        s=45,
    )

    unique_x = np.unique(scatter_df["graph_num_nodes"].to_numpy(dtype=float))
    if scatter_df.shape[0] >= 2 and unique_x.size >= 2:
        coeff = np.polyfit(
            scatter_df["graph_num_nodes"].to_numpy(dtype=float),
            scatter_df["graph_edge_var_mean"].to_numpy(dtype=float),
            1,
        )
        xs = np.linspace(float(scatter_df["graph_num_nodes"].min()), float(scatter_df["graph_num_nodes"].max()), 120)
        ys = coeff[0] * xs + coeff[1]
        ax.plot(xs, ys, color=COLOR_YELLOW, linewidth=2.0, alpha=0.95)

    ax.set_title("Edge variance vs graph size", loc="left")
    ax.set_xlabel("Graph size (number of nodes)")
    ax.set_ylabel("Mean edge variance per graph")
    ax.grid(alpha=0.3, color=COLOR_GREY)

    fig.tight_layout()
    scatter_path = PLOTS_DIR / "edge_variance_vs_graph_size_scatter.png"
    fig.savefig(scatter_path, dpi=int(FIG_DPI), bbox_inches="tight")
    print(f"Saved: {scatter_path}")
    plt.show()


In [ ]:
print("Smoke checks")

expected_graphs = int(len(eval_graph_indices))
actual_graphs = int(graph_edge_variance_df["graph_idx"].nunique()) if not graph_edge_variance_df.empty else 0
print(f"Graphs selected (scope): {expected_graphs}")
print(f"Graphs with edge variance rows: {actual_graphs}")

if explanation_runs_df.empty:
    print("No explanation rows found. Check configuration or runtime errors.")
else:
    run_counts = explanation_runs_df.groupby(["graph_idx", "target_node_type", "node_idx"])["seed"].nunique()
    print(f"Per-node unique seed runs: min={int(run_counts.min())}, max={int(run_counts.max())}, mean={float(run_counts.mean()):.2f}")

    incomplete = run_counts[run_counts < int(REPEATS)]
    if incomplete.empty:
        print("All explained nodes have full repeat coverage.")
    else:
        print(f"Nodes with incomplete repeat coverage: {int(len(incomplete))}")

required_files = [
    OUTPUT_DIR / "explanation_runs_raw.csv",
    OUTPUT_DIR / "node_level_variance.csv",
    OUTPUT_DIR / "feature_variance_by_target.csv",
    OUTPUT_DIR / "graph_edge_variance.csv",
    OUTPUT_DIR / "run_config.json",
    PLOTS_DIR / "edge_variance_histogram.png",
    PLOTS_DIR / "edge_variance_vs_graph_size_scatter.png",
]

for path in required_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:>7}  {path}")

if not failures_df.empty:
    print("Top failure reasons:")
    display(failures_df["error"].value_counts().head(10).to_frame("count"))
